# 03 — Train Stage2

In [1]:
from pathlib import Path
import subprocess
import sys

import torch
from google.colab import drive


drive.mount("/content/drive")

project_dir = Path("/content/drive/MyDrive/Research/PUMA/Exploration_20_Crop112-224_Graph224_G3_vs_GAT")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(project_dir / "requirements.txt")],
    check=True,
)

sys.path.insert(0, str(project_dir))

from common.settings import paths as get_paths, plans

paths = get_paths()
assert torch.cuda.is_available(), "Select a Colab GPU runtime."

print("GPU:", torch.cuda.get_device_name(0))
print("Stage 2 cache:", paths["stage2_cache"])

Mounted at /content/drive
GPU: NVIDIA L4
Stage 2 cache: /content/drive/MyDrive/Research/PUMA/Exploration_20_Crop112-224_Graph224_G3_vs_GAT/preprocess/stage2


## Verify feature cache


In [2]:
import json


manifest = json.loads((paths["stage2_cache"] / "manifest.json").read_text())

assert manifest["nuclei"] == 97429
assert manifest["rois"] == 206

print("Stage 2 feature cache:", manifest)

Stage 2 feature cache: {'rois': 206, 'names': ['training_set_metastatic_roi_001', 'training_set_metastatic_roi_002', 'training_set_metastatic_roi_003', 'training_set_metastatic_roi_004', 'training_set_metastatic_roi_005', 'training_set_metastatic_roi_006', 'training_set_metastatic_roi_007', 'training_set_metastatic_roi_008', 'training_set_metastatic_roi_009', 'training_set_metastatic_roi_010', 'training_set_metastatic_roi_011', 'training_set_metastatic_roi_012', 'training_set_metastatic_roi_013', 'training_set_metastatic_roi_014', 'training_set_metastatic_roi_015', 'training_set_metastatic_roi_016', 'training_set_metastatic_roi_017', 'training_set_metastatic_roi_018', 'training_set_metastatic_roi_019', 'training_set_metastatic_roi_020', 'training_set_metastatic_roi_021', 'training_set_metastatic_roi_022', 'training_set_metastatic_roi_023', 'training_set_metastatic_roi_024', 'training_set_metastatic_roi_025', 'training_set_metastatic_roi_026', 'training_set_metastatic_roi_027', 'trainin

## Train/skip both graph variants (2 variants × 2 plans × 4 folds = 16 runs)


In [3]:
import gc

from common.local_cache import local_copy_if_space
from stage2.train import train_one
from stage2.settings import VARIANTS


training_cache = local_copy_if_space(
    paths["stage2_cache"],
    "/content/puma_stage2_cache",
    buffer_gib=3,
)

EPOCHS = 20

for plan in plans:
    for fold in range(4):
        for mode in VARIANTS:
            print(f"EXPLORATION 20 | {mode} | {plan} | fold {fold}", flush=True)
            train_one(
                training_cache,
                paths["results"],
                plan,
                fold,
                epochs=EPOCHS,
                batch_size=0,
                mode=mode,
            )
            gc.collect()
            torch.cuda.empty_cache()


Copied preprocessing cache to Colab: /content/puma_stage2_cache
EXPLORATION 20 | G3 | A | fold 0
Exploration 20 G3 A/fold0 epoch 1/20 train_macro_f1=0.1511 val_macro_f1=0.1495 best=0.1495
Exploration 20 G3 A/fold0 epoch 2/20 train_macro_f1=0.3450 val_macro_f1=0.3405 best=0.3405
Exploration 20 G3 A/fold0 epoch 3/20 train_macro_f1=0.5729 val_macro_f1=0.5695 best=0.5695
Exploration 20 G3 A/fold0 epoch 4/20 train_macro_f1=0.6135 val_macro_f1=0.5966 best=0.5966
Exploration 20 G3 A/fold0 epoch 5/20 train_macro_f1=0.6332 val_macro_f1=0.6030 best=0.6030
Exploration 20 G3 A/fold0 epoch 6/20 train_macro_f1=0.6554 val_macro_f1=0.6158 best=0.6158
Exploration 20 G3 A/fold0 epoch 7/20 train_macro_f1=0.6731 val_macro_f1=0.6215 best=0.6215
Exploration 20 G3 A/fold0 epoch 8/20 train_macro_f1=0.6879 val_macro_f1=0.6233 best=0.6233
Exploration 20 G3 A/fold0 epoch 9/20 train_macro_f1=0.7018 val_macro_f1=0.6416 best=0.6416
Exploration 20 G3 A/fold0 epoch 10/20 train_macro_f1=0.7116 val_macro_f1=0.6380 best

## Final out-of-fold ROI F1 comparison: G3 vs GAT


In [4]:
from stage2.comparison import compare

comparison = compare(paths["results"])
print("Exploration 20 winner:", comparison["winner"])



EXPLORATION 20 | SHARED FOV112/224 | GRAPH R224 | ROI F1 (OOF)
Evaluation at GT nucleus centers: not end-to-end detector performance.
Model checkpoints selected by original validation nucleus Macro-F1.
----------------------------------------------------------------------------
Variant          Plan A ROI F1   Plan B ROI F1           MEAN
G3                    0.850268        0.847422       0.848845
GAT                   0.846348        0.841603       0.843976
----------------------------------------------------------------------------
WINNER (higher mean of Plan A/B ROI F1): G3
GAT - G3 = -0.004869
Saved: /content/drive/MyDrive/Research/PUMA/Exploration_20_Crop112-224_Graph224_G3_vs_GAT/results/stage2/exploration20_comparison.json
Exploration 20 winner: G3
